In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
    ElementClickInterceptedException
)
import time
import json


URL = "https://kr.tradingview.com/symbols/NASDAQ-RXRX/documents/"


from selenium import webdriver

def setup_driver(headless=False):
    options = webdriver.ChromeOptions()

    # 이미 떠 있는 Chrome에 접속
    options.debugger_address = "127.0.0.1:9222"

    driver = webdriver.Chrome(options=options)
    return driver


def safe_click(driver, element):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
    time.sleep(0.5)
    try:
        element.click()
    except (ElementClickInterceptedException, StaleElementReferenceException):
        driver.execute_script("arguments[0].click();", element)


def close_modal_if_exists(driver):
    """
    상세 transcript 창 닫기
    페이지 구조가 바뀔 수 있으므로 닫기 버튼을 여러 방식으로 시도
    """
    candidates = [
        "//button[@aria-label='Close']",
        "//button[contains(@class, 'close')]",
        "//div[@role='dialog']//button",
        "/html/body/div[8]/div[2]/div/div[1]/div/div/div/div/div[1]//button"
    ]

    for xpath in candidates:
        try:
            btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            safe_click(driver, btn)
            time.sleep(1)
            return True
        except Exception:
            continue

    # ESC로 닫기 시도
    try:
        from selenium.webdriver.common.keys import Keys
        body = driver.find_element(By.TAG_NAME, "body")
        body.send_keys(Keys.ESCAPE)
        time.sleep(1)
        return True
    except Exception:
        return False


def get_2026_section_container(driver):
    """
    2026 헤더를 찾고, 그 아래 article 목록이 들어있는 컨테이너를 반환
    """
    wait = WebDriverWait(driver, 20)

    year_el = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//*[normalize-space(text())='2026']")
        )
    )

    # '2026' 바로 아래 article 들이 들어있는 형제 컨테이너 탐색
    # 구조 변화에 덜 민감하게 작성
    container = year_el.find_element(
        By.XPATH,
        "./following-sibling::*[1]"
    )
    return container


def get_2026_articles_info(driver):
    """
    2026 섹션 내 article 제목/버튼 정보를 수집
    """
    container = get_2026_section_container(driver)

    articles = container.find_elements(By.XPATH, ".//article")
    results = []

    for idx, article in enumerate(articles, start=1):
        try:
            title = article.find_element(By.XPATH, ".//div[1]").text.strip()
        except NoSuchElementException:
            title = f"untitled_{idx}"

        # 텍스트 버튼: 보통 Event transcript / Call transcript
        buttons = article.find_elements(By.XPATH, ".//button")
        text_button = None

        for btn in buttons:
            label = btn.text.strip().lower()
            if "transcript" in label or "call" in label or "event" in label:
                text_button = btn
                break

        if text_button is None and buttons:
            text_button = buttons[0]

        results.append({
            "index": idx,
            "title": title,
            "button_xpath_relative_hint": ".//button",
        })

    return results


def open_article_by_index(driver, article_index):
    """
    2026 섹션 다시 찾고, 해당 순번 article의 transcript 버튼 클릭
    모달 재오픈 때문에 매번 다시 찾는 방식으로 작성
    """
    container = get_2026_section_container(driver)
    articles = container.find_elements(By.XPATH, ".//article")

    if article_index > len(articles):
        raise IndexError(f"article_index {article_index} out of range")

    article = articles[article_index - 1]

    title = article.find_element(By.XPATH, ".//div[1]").text.strip()

    buttons = article.find_elements(By.XPATH, ".//button")
    target_btn = None

    for btn in buttons:
        label = btn.text.strip().lower()
        if "transcript" in label or "call" in label or "event" in label:
            target_btn = btn
            break

    if target_btn is None:
        if not buttons:
            raise NoSuchElementException(f"버튼 없음: {title}")
        target_btn = buttons[0]

    safe_click(driver, target_btn)
    return title


def expand_full_text(driver):
    """
    상세 창에서 '전체 텍스트 펼치기' 역할의 버튼 클릭
    네가 준 XPath:
    /html/body/div[8]/div[2]/div/div[1]/div/div/div/div/div[2]/div/div[2]/div/div/button[2]
    이 구조를 우선 시도하고, 실패 시 일반화된 선택자로 대체
    """
    candidate_xpaths = [
        "/html/body/div[8]/div[2]/div/div[1]/div/div/div/div/div[2]/div/div[2]/div/div/button[2]",
        "//div[@role='dialog']//button[2]",
        "//button[contains(., '더 보기')]",
        "//button[contains(., 'Show more')]",
        "//button[contains(., '전체')]",
    ]

    for xpath in candidate_xpaths:
        try:
            btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            safe_click(driver, btn)
            time.sleep(1)
            return True
        except Exception:
            continue

    return False


def find_transcript_article(driver):
    """
    상세 본문 article 찾기
    네 절대 XPath 대신 //article 우선 사용
    """
    wait = WebDriverWait(driver, 20)

    # 모달/패널 내부 article 대기
    article = wait.until(
        EC.presence_of_element_located((By.XPATH, "//article"))
    )
    return article


def parse_paragraph(p_element):
    """
    strong/span 은 발표자, 나머지는 본문으로 간주
    결과는 Markdown 볼드 유지 형식
    """
    full_text = p_element.text.strip()
    if not full_text:
        return None

    speaker = ""
    try:
        speaker = p_element.find_element(By.XPATH, ".//strong/span").text.strip()
    except NoSuchElementException:
        pass

    if speaker:
        if full_text.startswith(speaker):
            body = full_text[len(speaker):].strip()
        else:
            body = full_text
        return f"**{speaker}** {body}".strip()

    return full_text


def extract_transcript_lines(driver):
    article = find_transcript_article(driver)
    paragraphs = article.find_elements(By.XPATH, ".//p")

    lines = []
    for p in paragraphs:
        line = parse_paragraph(p)
        if line:
            lines.append(line)

    return lines


def save_txt(all_docs, filename="rxrx_2026_transcripts.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for doc in all_docs:
            f.write(f"# {doc['title']}\n")
            f.write(f"Date: {doc.get('date', '')}\n\n")
            for line in doc["lines"]:
                f.write(line + "\n\n")
            f.write("\n" + "=" * 100 + "\n\n")


def save_json(all_docs, filename="rxrx_2026_transcripts.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(all_docs, f, ensure_ascii=False, indent=2)


def crawl_all_2026_transcripts(headless=False):
    driver = setup_driver(headless=headless)
    all_docs = []

    try:
        driver.get(URL)
        time.sleep(3)

        # 2026 article 개수/제목 먼저 확인
        article_infos = get_2026_articles_info(driver)
        print(f"2026 문서 수: {len(article_infos)}")

        for info in article_infos:
            idx = info["index"]
            print(f"\n[{idx}] 수집 시작: {info['title']}")

            try:
                title = open_article_by_index(driver, idx)
                time.sleep(2)

                expand_full_text(driver)
                lines = extract_transcript_lines(driver)

                doc = {
                    "year": 2026,
                    "index": idx,
                    "title": title,
                    "lines": lines,
                    "paragraph_count": len(lines),
                }
                all_docs.append(doc)

                print(f" -> 문단 수집 완료: {len(lines)}개")

            except Exception as e:
                print(f" -> 실패: {info['title']} | {e}")

            finally:
                close_modal_if_exists(driver)
                time.sleep(1)

        save_txt(all_docs)
        save_json(all_docs)

        print("\n저장 완료:")
        print("- rxrx_2026_transcripts.txt")
        print("- rxrx_2026_transcripts.json")

        return all_docs

    finally:
        driver.quit()


if __name__ == "__main__":
    docs = crawl_all_2026_transcripts(headless=False)

KeyboardInterrupt: 